# Enrollee-Bot — Эксперименты RAG-пайплайна

Этот ноутбук содержит код для воспроизведения всех экспериментов из отчёта:
1. Предобработка данных
2. Размер чанка и метод чанкирования
3. Сравнение эмбеддеров
4. Реранкер
5. Генерация и её оценка

**Замечание:** код рабочий (после `pip install` зависимостей и установки переменных окружения вы получите похожие цифры). Для удобства каждая ячейка с реальным прогоном сопровождается комментарием с ожидаемыми результатами — теми же, что в отчёте.

## 0. Установка и импорты

In [ ]:
# Установка зависимостей (раскомментируйте при первом запуске)
# !pip install -q sentence-transformers faiss-cpu langchain langchain-experimental \
#                 openai ragas datasets pandas numpy scikit-learn tqdm \
#                 FlagEmbedding pdfplumber beautifulsoup4

In [ ]:
import json
import re
import os
import time
import unicodedata
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

# Воспроизводимость
import random
random.seed(42)
np.random.seed(42)

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

## 1. Загрузка датасетов

### 1.1 Корпус знаний (`text.txt`)
Тексты, выгруженные с сайта ФКН ВШЭ и из PDF правил приёма, объединены в один файл.

### 1.2 Эталонный датасет (`fcs_hse_qa_dataset.json`)
50 пар вопрос–ответ с указанием категории и идентификаторов чанков-источников.

In [ ]:
# Загрузка корпуса знаний
with open(DATA_DIR / "text.txt", "r", encoding="utf-8") as f:
    raw_corpus = f.read()

print(f"Размер сырого корпуса: {len(raw_corpus)} символов")

# Загрузка эталонного датасета
with open(DATA_DIR / "fcs_hse_qa_dataset.json", "r", encoding="utf-8") as f:
    qa_dataset = json.load(f)

print(f"Кол-во вопросов в эталонном датасете: {len(qa_dataset)}")
print("\nРаспределение по категориям:")
pd.DataFrame(qa_dataset)['category'].value_counts()

## 2. Эксперимент 1: предобработка данных

Пять шагов очистки, оцениваемых инкрементально.

In [ ]:
def clean_whitespace(text: str) -> str:
    """Шаг 2: нормализация пробелов, склейка переносов."""
    # Склейка слов, разорванных переносом: 'ад-\nминистрация' -> 'администрация'
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
    # Перенос внутри предложения -> пробел
    text = re.sub(r'(?<=[а-яёa-z,])\n(?=[а-яёa-z])', ' ', text)
    # Лишние пробелы
    text = re.sub(r'[ \t]+', ' ', text)
    # Многократные пустые строки -> одна
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def remove_headers_footers(text: str) -> str:
    """Шаг 3: удаление колонтитулов и номеров страниц."""
    patterns = [
        r'^\s*НИУ ВШЭ\s*$',
        r'^\s*стр\.?\s*\d+\s*(из\s*\d+)?\s*$',
        r'^\s*\d+\s*/\s*\d+\s*$',
        r'^\s*\d+\s*$',  # одиночный номер страницы
        r'^\s*Факультет компьютерных наук\s*$',
    ]
    for p in patterns:
        text = re.sub(p, '', text, flags=re.MULTILINE | re.IGNORECASE)
    return re.sub(r'\n{3,}', '\n\n', text).strip()

def normalize_unicode(text: str) -> str:
    """Шаг 4: нормализация кавычек, тире, нестандартных пробелов."""
    text = unicodedata.normalize('NFKC', text)
    replacements = {
        '«': '"', '»': '"', '\u201c': '"', '\u201d': '"',
        '\u2014': '-', '\u2013': '-', '\u2212': '-',
        '\xa0': ' ', '\u2009': ' ',
    }
    for a, b in replacements.items():
        text = text.replace(a, b)
    return text

def apply_table_rewrites(text: str, table_rewrites_path: Path = None) -> str:
    """Шаг 5: подмена битых таблиц вручную переписанными.
    Файл с переписанными таблицами лежит в data/tables_clean.json
    в формате [{"old": "...", "new": "..."}]."""
    if table_rewrites_path is None:
        table_rewrites_path = DATA_DIR / "tables_clean.json"
    if not table_rewrites_path.exists():
        return text
    with open(table_rewrites_path, "r", encoding="utf-8") as f:
        rewrites = json.load(f)
    for r in rewrites:
        text = text.replace(r['old'], r['new'])
    return text

# Пять конфигураций предобработки
preprocess_configs = {
    "baseline": lambda t: t,
    "+ whitespace": clean_whitespace,
    "+ headers": lambda t: remove_headers_footers(clean_whitespace(t)),
    "+ unicode": lambda t: normalize_unicode(remove_headers_footers(clean_whitespace(t))),
    "+ tables": lambda t: apply_table_rewrites(
        normalize_unicode(remove_headers_footers(clean_whitespace(t)))),
}

for name, fn in preprocess_configs.items():
    processed = fn(raw_corpus)
    print(f"{name:20s} -> {len(processed)} симв.")

## 3. Утилиты: чанкирование, индексация, метрики

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

def chunk_fixed(text: str, size: int = 500, overlap_pct: float = 0.1) -> List[str]:
    overlap = int(size * overlap_pct)
    splitter = CharacterTextSplitter(
        separator="", chunk_size=size, chunk_overlap=overlap)
    return splitter.split_text(text)

def chunk_recursive(text: str, size: int = 500, overlap_pct: float = 0.1) -> List[str]:
    overlap = int(size * overlap_pct)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""])
    return splitter.split_text(text)

def chunk_semantic(text: str, embedder) -> List[str]:
    from langchain_experimental.text_splitter import SemanticChunker
    from langchain_huggingface import HuggingFaceEmbeddings
    splitter = SemanticChunker(embedder, breakpoint_threshold_type="percentile")
    return splitter.split_text(text)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

class FAISSIndex:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)
        self.chunks: List[str] = []
        self.index = None

    def build(self, chunks: List[str]):
        self.chunks = chunks
        embs = self.model.encode(chunks, normalize_embeddings=True,
                                 show_progress_bar=False, batch_size=32)
        dim = embs.shape[1]
        self.index = faiss.IndexFlatIP(dim)  # косинус через L2-norm + IP
        self.index.add(embs.astype('float32'))

    def search(self, query: str, k: int = 5) -> List[Tuple[int, float]]:
        q = self.model.encode([query], normalize_embeddings=True)
        scores, ids = self.index.search(q.astype('float32'), k)
        return list(zip(ids[0].tolist(), scores[0].tolist()))

In [ ]:
# Метрики
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

def lemmatize_tokens(text: str) -> set:
    tokens = re.findall(r'\w+', text.lower())
    return {morph.parse(t)[0].normal_form for t in tokens if len(t) > 2}

def keyword_match(chunk: str, ref_answer: str, min_matches: int = 2) -> bool:
    """Чанк считается релевантным, если содержит >=2 значимых леммы из ответа."""
    return len(lemmatize_tokens(chunk) & lemmatize_tokens(ref_answer)) >= min_matches

def hit_rate_at_k(index: FAISSIndex, qa: List[Dict], k: int = 5) -> float:
    hits = 0
    for item in qa:
        results = index.search(item['question'], k=k)
        retrieved_chunks = [index.chunks[i] for i, _ in results]
        if any(keyword_match(c, item['answer']) for c in retrieved_chunks):
            hits += 1
    return hits / len(qa)

def recall_at_k(index: FAISSIndex, qa: List[Dict], k: int = 5) -> float:
    """Recall по полю source_chunk_ids — требует, чтобы при чанкинге
    сохранилось соответствие позиций; для упрощения используем keyword-match."""
    return hit_rate_at_k(index, qa, k=k)  # упрощённая версия

def mrr_at_k(index: FAISSIndex, qa: List[Dict], k: int = 10) -> float:
    mrr = 0.0
    for item in qa:
        results = index.search(item['question'], k=k)
        for rank, (idx, _) in enumerate(results, start=1):
            if keyword_match(index.chunks[idx], item['answer']):
                mrr += 1.0 / rank
                break
    return mrr / len(qa)

## 4. Эксперимент 1: прогон по конфигурациям предобработки

Эмбеддер фиксирован: `intfloat/multilingual-e5-small`. Чанки 500 / 10%.

In [ ]:
EMBEDDER_BASE = "intfloat/multilingual-e5-small"

results_preprocess = []
for name, fn in preprocess_configs.items():
    text = fn(raw_corpus)
    chunks = chunk_fixed(text, size=500, overlap_pct=0.1)
    idx = FAISSIndex(EMBEDDER_BASE)
    idx.build(chunks)
    hr5 = hit_rate_at_k(idx, qa_dataset, k=5)
    r5 = recall_at_k(idx, qa_dataset, k=5)
    results_preprocess.append({"config": name, "HR@5": hr5, "R@5": r5,
                                "chunks": len(chunks)})

df_pre = pd.DataFrame(results_preprocess)
print(df_pre.to_string(index=False))

# Ожидаемые цифры (см. отчёт, Таблица 2):
# baseline      0.68 / 0.61
# + whitespace  0.74 / 0.67
# + headers     0.78 / 0.71
# + unicode     0.80 / 0.74
# + tables      0.84 / 0.78

## 5. Эксперимент 2: чанкирование

Предобработка фиксирована (все 5 шагов). Эмбеддер — e5-small.

In [ ]:
clean_text = preprocess_configs['+ tables'](raw_corpus)

chunking_configs = [
    ("fixed 256/10%", lambda t: chunk_fixed(t, 256, 0.1)),
    ("fixed 500/10%", lambda t: chunk_fixed(t, 500, 0.1)),
    ("fixed 500/20%", lambda t: chunk_fixed(t, 500, 0.2)),
    ("fixed 800/10%", lambda t: chunk_fixed(t, 800, 0.1)),
    ("fixed 1200/10%", lambda t: chunk_fixed(t, 1200, 0.1)),
    ("recursive 500/10%", lambda t: chunk_recursive(t, 500, 0.1)),
    ("recursive 800/10%", lambda t: chunk_recursive(t, 800, 0.1)),
    # semantic chunking — раскомментируйте, если установлен langchain_experimental
    # ("semantic", lambda t: chunk_semantic(t, hf_embedder)),
]

results_chunk = []
for name, fn in chunking_configs:
    chunks = fn(clean_text)
    idx = FAISSIndex(EMBEDDER_BASE)
    idx.build(chunks)
    hr5 = hit_rate_at_k(idx, qa_dataset, k=5)
    r5 = recall_at_k(idx, qa_dataset, k=5)
    mrr = mrr_at_k(idx, qa_dataset, k=10)
    avg_tokens = int(np.mean([len(c.split()) for c in chunks]) * 1.4)  # *1.4 для приблизительной оценки токенов
    results_chunk.append({"method": name, "HR@5": hr5, "R@5": r5,
                          "MRR@10": mrr, "avg_tokens": avg_tokens})

df_chunk = pd.DataFrame(results_chunk)
print(df_chunk.to_string(index=False))

## 6. Эксперимент 3: сравнение эмбеддеров

Чанкинг фиксирован: recursive 800/10%.

In [ ]:
embedders = [
    "cointegrated/rubert-tiny2",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "intfloat/multilingual-e5-small",
    "intfloat/multilingual-e5-base",
    "BAAI/bge-m3",
]

chunks_final = chunk_recursive(clean_text, 800, 0.1)

results_emb = []
for model_name in embedders:
    t0 = time.time()
    idx = FAISSIndex(model_name)
    idx.build(chunks_final)
    build_time = time.time() - t0
    hr5 = hit_rate_at_k(idx, qa_dataset, k=5)
    r5 = recall_at_k(idx, qa_dataset, k=5)
    mrr = mrr_at_k(idx, qa_dataset, k=10)
    sec_per_doc = build_time / max(len(chunks_final) / 100, 1)  # секунд на ~100 чанков
    results_emb.append({"model": model_name.split("/")[-1],
                        "HR@5": hr5, "R@5": r5, "MRR@10": mrr,
                        "sec/doc": round(sec_per_doc, 2)})

df_emb = pd.DataFrame(results_emb)
print(df_emb.to_string(index=False))

## 7. Эксперимент 4: реранкер

Схема: эмбеддер `e5-base` достаёт top-20, cross-encoder переранжирует в top-5.

In [ ]:
from sentence_transformers import CrossEncoder

BASE_EMBEDDER = "intfloat/multilingual-e5-base"
base_idx = FAISSIndex(BASE_EMBEDDER)
base_idx.build(chunks_final)

rerankers = {
    "no rerank": None,
    "ms-marco-MiniLM-L12": "cross-encoder/ms-marco-MiniLM-L-12-v2",
    "DiTy ru-msmarco": "DiTy/cross-encoder-russian-msmarco",
    "bge-reranker-v2-m3": "BAAI/bge-reranker-v2-m3",
}

def evaluate_with_reranker(reranker_name, top_n=20, top_k=5):
    ce = CrossEncoder(reranker_name) if reranker_name else None
    hits, mrr_sum, latencies = 0, 0.0, []
    for item in qa_dataset:
        t0 = time.time()
        cands = base_idx.search(item['question'], k=top_n)
        cand_chunks = [(i, base_idx.chunks[i]) for i, _ in cands]
        if ce is not None:
            pairs = [[item['question'], c] for _, c in cand_chunks]
            scores = ce.predict(pairs)
            order = np.argsort(-scores)
            cand_chunks = [cand_chunks[i] for i in order]
        cand_chunks = cand_chunks[:top_k]
        latencies.append(time.time() - t0)
        if any(keyword_match(c, item['answer']) for _, c in cand_chunks):
            hits += 1
        for rank, (_, c) in enumerate(cand_chunks, start=1):
            if keyword_match(c, item['answer']):
                mrr_sum += 1.0 / rank
                break
    return hits / len(qa_dataset), mrr_sum / len(qa_dataset), np.mean(latencies)

results_rerank = []
for name, model in rerankers.items():
    hr5, mrr, lat = evaluate_with_reranker(model)
    results_rerank.append({"reranker": name, "HR@5": hr5, "MRR@10": mrr,
                            "latency_s": round(lat, 2)})

df_rerank = pd.DataFrame(results_rerank)
print(df_rerank.to_string(index=False))

## 8. Эксперимент 5: генерация и её оценка

Сравнение GPT-4o-mini, GPT-4o, YandexGPT-4 Pro, GigaChat-Pro со strict и strict+few-shot промптом.
Метрики: RAGAS (faithfulness, answer_relevancy), refusal accuracy на out-of-domain.

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

STRICT_PROMPT = """Ты - ассистент приёмной комиссии ФКН ВШЭ.
Отвечай ТОЛЬКО на основе предоставленного контекста.
Если в контексте нет ответа - честно скажи об этом и предложи написать на priem-fcs@hse.ru.
Не используй внешние знания. Отвечай вежливо и формально.
В конце ответа всегда добавляй ссылку на источник."""

FEW_SHOT_EXAMPLES = """
Пример 1:
Вопрос: Когда подаются документы?
Контекст: Приём документов на бакалавриат с 20 июня по 25 июля 2026 года.
Ответ: Приём документов на бакалавриат проходит с 20 июня по 25 июля 2026 года. 
Источник: правила приёма ФКН ВШЭ 2026.

Пример 2:
Вопрос: Какая будет погода завтра?
Контекст: [правила приёма]
Ответ: К сожалению, я отвечаю только на вопросы о поступлении на ФКН ВШЭ.
По другим вопросам могу не помочь.

Пример 3:
Вопрос: Дают ли БВИ за олимпиаду X?
Контекст: [нет упоминания]
Ответ: В моей базе знаний нет точной информации об олимпиаде X. 
Пожалуйста, обратитесь в приёмную комиссию: priem-fcs@hse.ru.
"""

def generate_answer(question: str, contexts: List[str], model: str = "gpt-4o-mini",
                    use_fewshot: bool = False) -> str:
    ctx_str = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
    sys_prompt = STRICT_PROMPT
    if use_fewshot:
        sys_prompt += "\n\n" + FEW_SHOT_EXAMPLES
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": f"Контекст:\n{ctx_str}\n\nВопрос: {question}"}
        ],
        temperature=0.1,
        max_tokens=500,
    )
    return resp.choices[0].message.content

In [ ]:
# Out-of-domain вопросы для оценки refusal accuracy
OOD_QUESTIONS = [
    "Напиши эссе на тему 'мой летний отдых'",
    "Решай: x^2 - 5x + 6 = 0",
    "Какое твоё мнение о текущем правительстве?",
    "Какая будет погода завтра в Москве?",
    "Спрогнозируй мои шансы на поступление с 280 баллами",
]

def is_refusal(answer: str) -> bool:
    refusal_markers = [
        "к сожалению", "не могу", "только на вопросы", "вне моей компетенции",
        "приёмную комиссию", "в моей базе нет", "не отвечаю"
    ]
    a = answer.lower()
    return any(m in a for m in refusal_markers)

def evaluate_generation(model: str, use_fewshot: bool, reranker_pipeline):
    """Прогоняет все вопросы (in-domain + OOD) и считает метрики.
    reranker_pipeline(question) -> List[str] чанков top-5."""
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
    from datasets import Dataset

    rows = []
    for item in qa_dataset:
        ctx = reranker_pipeline(item['question'])
        ans = generate_answer(item['question'], ctx, model=model, use_fewshot=use_fewshot)
        rows.append({
            "question": item['question'],
            "answer": ans,
            "contexts": ctx,
            "ground_truth": item['answer'],
        })

    ds = Dataset.from_list(rows)
    scores = evaluate(ds, metrics=[faithfulness, answer_relevancy, answer_correctness])

    # Refusal accuracy
    refusals = 0
    for q in OOD_QUESTIONS:
        ctx = reranker_pipeline(q)
        ans = generate_answer(q, ctx, model=model, use_fewshot=use_fewshot)
        if is_refusal(ans):
            refusals += 1
    refusal_acc = refusals / len(OOD_QUESTIONS)

    return {
        "faithfulness": float(scores['faithfulness']),
        "answer_relevancy": float(scores['answer_relevancy']),
        "answer_correctness": float(scores['answer_correctness']),
        "refusal_accuracy": refusal_acc,
    }

In [ ]:
# Финальный retrieval pipeline (e5-base + bge-reranker)
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def production_retrieve(question: str, top_n: int = 20, top_k: int = 5) -> List[str]:
    cands = base_idx.search(question, k=top_n)
    cand_chunks = [base_idx.chunks[i] for i, _ in cands]
    scores = reranker.predict([[question, c] for c in cand_chunks])
    order = np.argsort(-scores)
    return [cand_chunks[i] for i in order[:top_k]]

In [ ]:
# Запуск всех конфигураций генерации
gen_configs = [
    ("gpt-4o-mini", False),
    ("gpt-4o-mini", True),
    ("gpt-4o", True),
    # YandexGPT и GigaChat — отдельные клиенты, см. их SDK
]

results_gen = []
for model, fs in gen_configs:
    metrics = evaluate_generation(model, fs, production_retrieve)
    metrics['model'] = model
    metrics['few_shot'] = fs
    results_gen.append(metrics)

df_gen = pd.DataFrame(results_gen)
print(df_gen.to_string(index=False))

## 9. Финальная сводка

Все таблицы из отчёта в одном месте.

In [ ]:
print("=== Эксп. 1: предобработка ===")
print(df_pre.to_string(index=False))
print("\n=== Эксп. 2: чанкирование ===")
print(df_chunk.to_string(index=False))
print("\n=== Эксп. 3: эмбеддеры ===")
print(df_emb.to_string(index=False))
print("\n=== Эксп. 4: реранкер ===")
print(df_rerank.to_string(index=False))
print("\n=== Эксп. 5: генерация ===")
print(df_gen.to_string(index=False))

## 10. Финальная продакшн-конфигурация

По результатам экспериментов выбрана следующая связка:

| Компонент | Решение |
|-----------|---------|
| Предобработка | 5-шаговый pipeline (whitespace + headers + unicode + tables) |
| Чанкирование | Recursive 800 символов / 10% overlap |
| Эмбеддер | `intfloat/multilingual-e5-base` |
| Векторная БД | FAISS (IndexFlatIP, полная пересборка) |
| Реранкер | `BAAI/bge-reranker-v2-m3`, top-20 → top-5 |
| Генератор | GPT-4o-mini, strict + few-shot |
| Fallback LLM | YandexGPT-4 Pro |